# Gemini + RAG demo - reproducibility analysis of a single PDF

Uses `gemini_rag.GeminiPaperAnalyst` to extract a structured `PaperProfile` from any PDF on disk.

**Setup**

```bash
uv pip install google-genai pydantic
export GEMINI_API_KEY=your-key-here   # or GOOGLE_API_KEY
```

The pipeline runs five independent, schema-constrained queries against the PDF (methodology, datasets, artefacts, figures/tables, parameters) at `temperature=0` with a system instruction that pins answers to the attached document. Each extracted item carries a `source_quote` field so you can audit for hallucination.

## 0. Available paired papers

Lists every row of `data/rescience_bibtex_table.xlsx` that has a PDF in **both** the `RESCIENCE C` and `ORIGINAL` sections. Pick one of these filenames as the `PDF_PATH` in section 2.

In [1]:
import pandas as pd
from pathlib import Path

XLSX = Path("data/rescience_bibtex_table.xlsx")
RESCIENCE_CACHE = Path("data/pdf_cache")
ORIGINAL_CACHE = Path("data/pdf_original_cache")

# The xlsx has a two-row header: (section, field). Flatten to prefixed names so we
# can address every column unambiguously -- e.g. "rescience_filename", "original_filename".
df_all = pd.read_excel(XLSX, header=[0, 1])
def _flatten(col: tuple[str, str]) -> str:
    section, field = col
    if section.startswith("Unnamed") or field.startswith("Unnamed"):
        return "idx"
    prefix = {"RESCIENCE C": "rescience", "ORIGINAL": "original"}.get(section, section.lower())
    return f"{prefix}_{field}"
df_all.columns = [_flatten(c) for c in df_all.columns]

# Use the filename columns (authoritative) rather than the pdf_file download-marker columns.
def _resolve(cache: Path, name) -> Path | None:
    if not isinstance(name, str) or not name.strip() or name.strip().lower() == "nan":
        return None
    p = cache / name.strip()
    return p if p.is_file() else None

df_all["_rescience_path"] = df_all["rescience_filename"].map(lambda n: _resolve(RESCIENCE_CACHE, n))
df_all["_original_path"]  = df_all["original_filename"].map(lambda n: _resolve(ORIGINAL_CACHE, n))

paired = df_all[df_all["_rescience_path"].notna() & df_all["_original_path"].notna()].copy()
missing_rescience = df_all["_rescience_path"].isna().sum()
missing_original  = df_all["_original_path"].isna().sum()

paired_view = pd.DataFrame({
    "rescience_pdf":   paired["rescience_filename"].values,
    "original_pdf":    paired["original_filename"].values,
    "rescience_title": paired["rescience_title"].values,
    "original_title":  paired["original_title"].values,
})
print(f"Total rows in xlsx:            {len(df_all)}")
print(f"Rescience PDFs missing on disk: {missing_rescience}")
print(f"Original PDFs missing on disk:  {missing_original}")
print(f"Paired (both PDFs available):   {len(paired_view)}\n")
paired_view

Total rows in xlsx:            222
Rescience PDFs missing on disk: 0
Original PDFs missing on disk:  7
Paired (both PDFs available):   215



,rescience_pdf,original_pdf,rescience_title,original_title
0,2026_01_article.pdf,2026_01_article.pdf,Learning with Noisy Labels [Re]visited,Learning with Noisy Labels Revisited: A Study ...
1,2026_02_article.pdf,2026_02_article.pdf,[Re] Curve-Fitting with Piecewise Parametric C...,Curve-Fitting with Piecewise Parametric Cubics
2,2025_01_article.pdf,2025_01_article.pdf,[Re] Model of thalamocortical slow-wave sleep ...,Model of thalamocortical slow-wave sleep oscil...
3,2025_02_article.pdf,2025_02_article.pdf,[Re] Learning Fair Graph Representations via A...,Learning Fair Graph Representations via Automa...
4,2025_03_article.pdf,2025_03_article.pdf,[Re] Network Deconvolution,Network Deconvolution
...,...,...,...,...
210,2016_04_article.pdf,2016_04_article.pdf,[Re] Multiple dynamical modes of thalamic rela...,Multiple dynamical modes of thalamic relay neu...
211,2016_05_article.pdf,2016_05_article.pdf,[Re] Chaos in a long-term experiment with a pl...,Chaos in a long-term experiment with a plankto...
212,2016_06_article.pdf,2016_06_article.pdf,[Re] Least-cost modelling on irregular landsca...,Least-cost modelling on irregular landscape gr...
213,2016_07_article.pdf,2016_07_article.pdf,[Re] Speed/accuracy trade-off between the habi...,Speed/accuracy trade-off between the habitual ...


## 1. Imports and setup

In [2]:
import json
from pathlib import Path

from gemini_rag import GeminiPaperAnalyst, PaperProfile

## 2. Point at a specific PDF

Three common sources:
1. one of the paired RESCIENCE / ORIGINAL papers (rows of `paired_view` above) — block **(a)**;
2. an ad-hoc PDF dropped under `data/specific_cases/` — block **(b)** (default);
3. any absolute path — block **(c)**.

Section 0 only matters for source (1); skip it if you are analysing a specific-case or arbitrary PDF.

In [3]:
SPECIFIC_CASES = Path("data/specific_cases")

# ---- choose ONE of the three blocks below -----------------------------------
# (a) paired paper from data/pdf_original_cache (or RESCIENCE_CACHE):
#   PDF_NAME = paired_view.iloc[-1]["original_pdf"]
#   PDF_PATH = ORIGINAL_CACHE / PDF_NAME

# (b) specific case under data/specific_cases/:
PDF_PATH = SPECIFIC_CASES / "simple_paper.pdf"

# (c) any absolute path:
#   PDF_PATH = Path("/abs/path/to/paper.pdf")
# -----------------------------------------------------------------------------

assert PDF_PATH.exists(), f"missing: {PDF_PATH}"
print(f"PDF: {PDF_PATH}  ({PDF_PATH.stat().st_size / 1024:.1f} KB)")

PDF: data/specific_cases/simple_paper.pdf  (2238.5 KB)


## 3. Run the analyst

In [4]:
analyst = GeminiPaperAnalyst(model="gemini-3-flash-preview")
profile: PaperProfile = analyst.analyze(PDF_PATH)

Uploading simple_paper.pdf to Gemini Files API...
  querying: header ...
  querying: nodes_source ...
  querying: nodes_sink ...
  querying: nodes_process ...
  querying: artefacts ...
  querying: parameters ...


## 4. Inspect the PaperProfile

In [5]:
n_figs = sum(1 for s in profile.nodes_sink if s.type == "figure")
n_tabs = sum(1 for s in profile.nodes_sink if s.type == "table")
n_methods = sum(1 for p in profile.nodes_process if p.process_type == "method")
n_experiments = sum(1 for p in profile.nodes_process if p.process_type == "experiment")

m = profile.metadata
print(f"Title:   {m.title}")
print(f"Authors: {', '.join(m.authors)}")
print(f"Process steps:     {len(profile.nodes_process)}  (methods={n_methods}, experiments={n_experiments})")
print(f"Datasets:          {len(profile.nodes_source)}")
print(f"Figures:           {n_figs}")
print(f"Tables:            {n_tabs}")
print(f"Hyperparameters:   {len(m.hyperparameters)}")
print(f"Repository links:  {m.repository_links}")

Title:   A Deep Reinforcement Learning Approach for Ramp Metering Based on Traffic Video Data
Authors: Bing Liu, Yu Tang, Yuxiong Ji, Yu Shen, Yuchuan Du
Process steps:     4  (methods=3, experiments=1)
Datasets:          1
Figures:           8
Tables:            1
Hyperparameters:   8
Repository links:  ['https://arxiv.org/abs/2012.12104', 'https://arxiv.org/abs/1705.02755', 'https://arxiv.org/abs/1412.6980', 'https://arxiv.org/abs/1312.5602']


### Methodology steps (with grounding quotes)

In [6]:
for step in profile.nodes_process:
    print(f"[{step.process_type}] [{step.node_name}] {step.description}")
    if step.tools_mentioned:
        print(f"    tools: {', '.join(step.tools_mentioned)}")
    if step.source_quote:
        print(f'    quote: "{step.source_quote}"')
    print()

[method] [Vehicle Location Extraction] Extracts (xi, yi) coordinates from raw video frames to create position matrices mt of size X by Y.
    quote: "Vehicle locations are extracted from the traffic video frames and are reformed as position matrices."

[method] [State Representation Construction] Stacks N consecutive position matrices after down-sampling to represent the traffic state st.
    quote: "we stack the matrices of consecutive N time steps to represent the state st"

[method] [Deep Q-Network Training] Trains a CNN-based Q-network using experience replay, target networks, and multitask learning to predict Q-values, speed, and queue length.
    tools: Adam version not stated, SUMO version not stated
    quote: "The algorithm for training the ramp metering policy is presented in Algorithm 1."

[experiment] [Simulation Evaluation] Evaluates the trained policy in a SUMO simulation of a real-world freeway segment over 20 experiments.
    tools: SUMO version not stated, NVIDIA Quadr

### Datasets

In [7]:
for ds in profile.nodes_source:
    print(f"- {ds.node_name}  ({ds.availability.value})")
    if ds.url:
        print(f"    url: {ds.url}")
    if ds.source_quote:
        print(f'    quote: "{ds.source_quote}"')

- real-world traffic data  (upon_request)
    quote: "A series of simulation experiments based on real-world traffic data are conducted to evaluate the proposed approach."


## 5. Save the profile to JSON

In [8]:
import re

stem = PDF_PATH.stem  # e.g. "2026_01_article"
existing = list(PDF_PATH.parent.glob(f"{stem}_exe*.profile.json"))
used = [
    int(m.group(1))
    for p in existing
    if (m := re.match(rf"{re.escape(stem)}_exe(\d+)\.profile\.json$", p.name))
]
exe_n = (max(used) + 1) if used else 1

out = PDF_PATH.parent / f"{stem}_exe{exe_n}.profile.json"
out.write_text(profile.model_dump_json(indent=2))
print(f"wrote: {out}")

wrote: data/specific_cases/simple_paper_exe5.profile.json


## 6. Token usage and cost estimate

`analyst.usage_log` captures `usage_metadata` from every call since the last `analyze()`. `analyst.usage_summary()` aggregates it and applies the paid-tier rates in `gemini_rag.GEMINI_PRICING_USD_PER_MTOK`.

Rates drift - always cross-check against https://ai.google.dev/pricing and your AI Studio billing dashboard for authoritative numbers.

In [9]:
summary = analyst.usage_summary()

print(f"Model: {summary['model']}")
print(f"API calls: {summary['calls']}")
print()
print(f"{'query':<18} {'input':>10} {'output':>10}")
print("-" * 40)
for q in summary["per_query"]:
    print(f"{q['name']:<18} {q['input_tokens']:>10,} {q['output_tokens']:>10,}")
print("-" * 40)
print(f"{'TOTAL':<18} {summary['input_tokens']:>10,} {summary['output_tokens']:>10,}")
print(f"Grand total tokens: {summary['total_tokens']:,}")
print()
if summary["total_usd"] is None:
    print(f"No price table entry for model {summary['model']!r} - add one to GEMINI_PRICING_USD_PER_MTOK.")
else:
    print(
        f"Estimated cost (USD):  input ${summary['input_usd']:.6f}  "
        f"+  output ${summary['output_usd']:.6f}  "
        f"=  ${summary['total_usd']:.6f}"
    )

Model: gemini-3-flash-preview
API calls: 6

query                   input     output
----------------------------------------
header                  7,249         68
nodes_source            7,860        189
nodes_sink              8,640      1,192
nodes_process           9,314      1,336
artefacts               7,272        146
parameters              7,268        336
----------------------------------------
TOTAL                  47,603      3,267
Grand total tokens: 50,870

No price table entry for model 'gemini-3-flash-preview' - add one to GEMINI_PRICING_USD_PER_MTOK.
